# Which of the lab's stars feel a pull? Read the catalog of accelerations

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/gaia_02_who_feels_a_pull.ipynb)

Before fitting a star's motion, check whether there is anything to fit. In this notebook you look up four stars from the lab's own papers in the Hipparcos-Gaia Catalog of Accelerations, turn each one's score into a significance, and decide which stars show a pull from an unseen companion.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. The last cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# orbitize! is built from source in Colab, so this takes about two minutes the first time.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("orbitize") is None:
    %pip install -q "habtrain[orbits] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/gaia_02_who_feels_a_pull.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## What the catalog gives you

For each star the catalog lists three motions across the sky, each in two directions (east-west and north-south), in milliarcseconds per year:
- the motion Hipparcos measured, around 1991
- the motion Gaia measured, around 2016
- the average motion over the 25 years between, from the change in the star's position

A star with nothing pulling it has the same motion all three ways. The catalog compares them and gives one number, a **chi-square**, for how badly a steady motion fits. Two degrees of freedom: the two directions. A bigger chi-square means a stronger sign of a pull. To read it, turn it into a significance in sigma: the chance that noise alone gives a chi-square this big, expressed on the familiar bell-curve scale. Under 2 sigma is no clear sign; 3 sigma or more is a real pull.

The catalog ships inside orbitize!, so nothing downloads ([Brandt 2021](https://arxiv.org/abs/2105.11662)).

## Worked example: iota Draconis

iota Draconis is a giant star with a planet on an eccentric 511-day orbit, iota Dra b. The lab's paper on it found a second, much longer-period companion in the velocities, iota Dra c ([Hill et al. 2021](https://arxiv.org/abs/2107.13583)). Its Hipparcos number is 75458.

In [ ]:
# iota Dra's row in the catalog of accelerations.
row = ht.orbits.hgca_row(75458)
print(f"Chi-square {row['chi2']:.2f}, which is {ht.orbits.pull_sigma(row['chi2']):.1f} sigma")

# Its three motions in each direction, shown as differences from the 25-year average.
fig_iota = ht.orbits.plot_proper_motions(row, "iota Dra")
plt.show()

All three motions agree within their error bars, in both directions, and the chi-square is under 1 sigma. The catalog shows no clear pull on iota Dra. That does not mean there is no companion: it means any pull is too small for these measurements, which sets an upper limit on how heavy iota Dra c can be. The lab's paper says the same.

## Your turn: the lab's other stars

[Hill et al. 2020](https://arxiv.org/abs/2003.02385) refined the orbits of planets around three stars: **HD 9446**, **HD 43691** and **HD 179079**. For HD 9446 it checked the Hipparcos and Gaia motions with an earlier Gaia release (DR2) and found no pull (its section 4.2).

First find each star's Hipparcos number. Search the star's name on [SIMBAD](https://simbad.cds.unistra.fr/simbad/) and read the identifier that starts with HIP.

> **Write down: each star's Hipparcos number**
>
> - Put each star's Hipparcos number, as a whole number, in `HIP_HD9446`, `HIP_HD43691` and `HIP_HD179079`.
>
> **Where to look:** SIMBAD's identifiers list for each star: the one that starts with HIP.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

On SIMBAD, type the name in the basic search box. The identifiers are listed under the star's details, alphabetically; HIP is near the middle.

</details>

In [ ]:
import pandas as pd

HIP_HD9446 = ...     # whole numbers, from SIMBAD
HIP_HD43691 = ...
HIP_HD179079 = ...
HIP = {"HD 9446": HIP_HD9446, "HD 43691": HIP_HD43691, "HD 179079": HIP_HD179079}

# Look each star up in the catalog and work out its significance. iota Dra is added for comparison.
table = []
for name, hip in list(HIP.items()) + [("iota Dra", 75458)]:
    r = ht.orbits.hgca_row(hip) if ht.record.filled(hip) else None
    if r is None:
        print(f"{name}: no row found for {hip}. Check the number.")
        continue
    table.append({"star": name, "HIP": hip, "chi-square": round(r["chi2"], 2), "sigma": round(ht.orbits.pull_sigma(r["chi2"]), 2)})
table = pd.DataFrame(table)
table

> **Write down: which stars feel a pull**
>
> - For each star, your call: "no pull", "a hint" (2 to 3 sigma) or "a pull" (3 sigma or more). Put them in `CALL_HD9446`, `CALL_HD43691` and `CALL_HD179079`.
> - Does your HD 9446 call agree with the earlier check in Hill et al. 2020, which used Gaia DR2? One line. Put it in `HD9446_AGREES`.
> - If a star shows no pull, what does that still tell you about any companion it has? One line. Put it in `NO_PULL_MEANS`.
>
> **Where to look:** the table above, and the worked example.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
CALL_HD9446 = ...     # "no pull", "a hint" or "a pull"
CALL_HD43691 = ...
CALL_HD179079 = ...
CALLS = {"HD 9446": CALL_HD9446, "HD 43691": CALL_HD43691, "HD 179079": CALL_HD179079}
HD9446_AGREES = ...   # one line, in quotes
NO_PULL_MEANS = ...   # one line, in quotes

# Keep the answers, with the table the notebook worked out, for your hand-in.
ht.record.save("hip", HIP, "each star's Hipparcos number")
ht.record.save("pull_table", table, "chi-square and significance for each star")
ht.record.save("calls", CALLS, "your call for each star")
ht.record.save("hd9446_agrees", HD9446_AGREES, "whether HD 9446 agrees with the 2020 check")
ht.record.save("no_pull_means", NO_PULL_MEANS, "what no pull still tells you")

### Hand in

Run the cell below. It shows all your answers in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `pull-table`, and download them.

In [ ]:
# Your answers and the iota Dra plot, saved as your hand-in.
ht.record.handin("pull-table", ["hip", "pull_table", "calls", "hd9446_agrees", "no_pull_means"], figures={"iota_dra_motions": fig_iota})